<a href="https://colab.research.google.com/github/hiteshraos/BIS-LAB/blob/main/1BM24CS117_CYBERSECURITY_CUCKOO_SEARCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import math
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Generate sample intrusion detection dataset
X, y = make_classification(n_samples=500, n_features=8,
                           n_informative=4, n_redundant=2,
                           random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25,
    random_state=42, stratify=y_train)

n = X.shape[1]

# Evaluate feature subset
def fitness(mask):
    if not mask.any():
        return 0

    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_fit[:, mask], y_fit)

    acc = accuracy_score(y_val, model.predict(X_val[:, mask]))
    return acc - 0.01 * mask.sum() / n

# Levy flight
def levy():
    beta = 1.5
    sigma = (math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
             (math.gamma((1 + beta) / 2) * beta *
              2 ** ((beta - 1) / 2))) ** (1 / beta)

    u = np.random.normal(0, sigma)
    v = np.random.normal()
    return u / abs(v) ** (1 / beta)

# Initialize nests
nests = np.random.randint(0, 2, (10, n))
nests[:, 0] = 1

scores = np.array([fitness(x) for x in nests])
best = nests[np.argmax(scores)].copy()
best_score = max(scores)

# Cuckoo Search iterations
for iteration in range(20):

    for i in range(10):
        step = levy()

        new = nests[i] + 0.5 * step * (nests[i] - best)
        new = (new > 0.5).astype(int)

        if not new.any():
            new[np.random.randint(n)] = 1

        score = fitness(new)

        if score > scores[i]:
            nests[i] = new
            scores[i] = score

        if score > best_score:
            best = new.copy()
            best_score = score

    # Abandon worst nests
    for i in np.argsort(scores)[:2]:
        nests[i] = np.random.randint(0, 2, n)

        if not nests[i].any():
            nests[i, np.random.randint(n)] = 1

        scores[i] = fitness(nests[i])

        if scores[i] > best_score:
            best = nests[i].copy()
            best_score = scores[i]

# Train final classifier
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train[:, best.astype(bool)], y_train)

accuracy = accuracy_score(
    y_test, model.predict(X_test[:, best.astype(bool)]))

print("Selected Features:",
      [f"F{i+1}" for i, v in enumerate(best) if v])
print("Number of Selected Features:", sum(best))
print("Classification Accuracy:", round(accuracy * 100, 2), "%")


Selected Features: ['F1', 'F2', 'F3', 'F5', 'F7', 'F8']
Number of Selected Features: 6
Classification Accuracy: 81.33 %
